第1步：单独执行，不要Run All。上传和封存后仍需在网页点击真实发送。

In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.util

if "board_session" in globals() or "ps_candidate_session" in sys.modules:
    raise RuntimeError("请使用全新内核，不复用旧会话")
root = Path("/home/xilinx/jupyter_notebooks/test_10_14")
candidate = root / "PYNQ_Image_Web_v0_2_1_2026-10-19"
host = candidate / "runtime" / "host"
rt_host = root / "PS_UART_RT_v5_2026-10-15" / "host"
native_library = rt_host / "libuartlite_service_v5_rt.so"
manifest = json.loads((candidate / "sha256.json").read_text(encoding="utf-8"))
for relative, expected in manifest.items():
    raw = (candidate / relative).read_bytes()
    assert len(raw) == expected["bytes"], relative
    assert hashlib.sha256(raw).hexdigest() == expected["sha256"], relative
print("候选文件校验通过:", len(manifest))
# Validate saved code before any Overlay/MMIO/UART/session allocation.
sys.path.insert(0,str(candidate))
import package_tools
assert Path(package_tools.__file__).resolve().parent==candidate.resolve()
print("Notebook代码检查:",json.dumps(package_tools.verify()["notebook"],ensure_ascii=False))
assert native_library.is_file(), "找不到 RT v5 接收库"
name = "h1_7020_evidence_candidate_v0_1"
for ext, digest in {
    ".bit": "0c0b686e80e37e49e64ab07b719a0c1ea3387451f3ceebdba5dcf38b8c68b61c",
    ".hwh": "8d65319fd06ccafb9acdabe1c19868106cd1945f085b2fa03d55b89d238bd2bc",
}.items():
    assert hashlib.sha256((root / (name + ext)).read_bytes()).hexdigest() == digest
for path in (str(rt_host), str(host), str(candidate)):
    if path in sys.path: sys.path.remove(path)
    sys.path.insert(0, path)
from pynq import Overlay, MMIO, Clocks
from ps_candidate_session import CandidateSession, BANKS
from ch1_uartlite_realtime_transport import RealtimeUARTLiteControlTransport
from ch1_admission_parts import _has_exact_rx
from controlled_trial_executor import ControlledTrialExecutor, TrialRequest
from local_rule_trial_loop import LocalRuleTrialLoop
from multipoint_capability_preflight import inspect_multipoint_capability
from fault_profile import FaultProfile
from fault_pilot_workflow import inspect_raw_mark_interface
for module_name in ("ps_candidate_session", "local_rule_trial_loop", "raw_mark_snapshot", "ch1_control_session"):
    assert Path(sys.modules[module_name].__file__).resolve().parent == host.resolve()
assert "uart_byte_exchange_saved" in _has_exact_rx.__code__.co_consts
spec = importlib.util.spec_from_file_location("unknown_candidate_probe", candidate / "read_only_probe.py")
probe_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(probe_module)
# Hold the same exclusive lock across Overlay reload and session creation.
import fcntl
runs = root / "board_runs"
assert runs.is_dir(), "保留原有 board_runs，不创建替代目录"
assert "board_lock" not in globals(), "不要重复运行初始化单元格"
board_lock = (runs / "exclusive.lock").open("a+")
try:
    fcntl.flock(board_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
except BaseException:
    board_lock.close()
    raise
overlay = Overlay(str(root / (name + ".bit")))
Clocks.fclk0_mhz = 150
print("FCLK0 配置读回 MHz:", Clocks.fclk0_mhz)
fresh = probe_module.probe(MMIO, root / (name + ".hwh"))
print(json.dumps(fresh, indent=2, ensure_ascii=False))
expected_status = dict(H1=1, CTRL=0, MAIN=0, AUX=0, MAP=0, CH0=8, PROFILE=4, SYS=0)
assert fresh["all_IDs_match"]
assert all(int(b["status"], 16) == expected_status[b["bank"]] for b in fresh["banks"])
assert fresh["local_epoch"] == fresh["retire_count"] == fresh["grant_count"] == 0
print("上传图片整图候选已加载；原Overlay配置后检查通过，尚未创建会话、未执行试验。")


第2步：单独执行，不要Run All。上传和封存后仍需在网页点击真实发送。

In [ ]:
import os, fcntl
if "board_session" in globals():
    raise RuntimeError("本内核已初始化过，不重复执行")
runs = root / "board_runs"
assert runs.is_dir()
assert "board_lock" in globals() and not board_lock.closed, "先完成第1步的独占检查"
fcntl.flock(board_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
alloc = runs / "namespace_allocations.jsonl"
assert alloc.is_file(), "禁止重建或重置历史编号文件"
numbers = [json.loads(line)["number"] for line in alloc.read_text(encoding="utf-8").splitlines() if line.strip()]
assert numbers == list(range(1, len(numbers) + 1))
n = len(numbers) + 1
assert n < 0x10000000
session_dir = runs / ("session_%08x" % n)
assert not session_dir.exists()
with alloc.open("a", encoding="utf-8") as f:
    f.write(json.dumps({"number": n}) + "\n")
    f.flush(); os.fsync(f.fileno())
directory_fd = os.open(str(runs), os.O_RDONLY)
try: os.fsync(directory_fd)
finally: os.close(directory_fd)
buses = {bank: MMIO(base, size) for bank, (base, size, _) in BANKS.items()}
board_session = CandidateSession(
    buses, session_dir, local_epoch=0x10000000+n, peer_epoch=0x20000000+n,
    session=0x30000000+n, control_epoch=0x40000000+n,
    exclusive_owner=True, poll_limit=10000)
control_uart = RealtimeUARTLiteControlTransport(
    MMIO(0x42c00000, 0x10000), board_session.journal,
    library=native_library, exclusive_owner=True, timeout=2.0,
    receiver_cpu=1, receiver_priority=10, receiver_idle_ns=50000)
try:
    control_uart.start_receiver()
    board_session.initialize(control_uart)
    print("READY:", board_session.upload.ready)
    print("会话故障:", board_session.fault)
    print("证据目录:", board_session.root)
    print("实际线程调度:")
    print(json.dumps(control_uart.scheduling_snapshot(), indent=2))
    mark_preflight = inspect_raw_mark_interface(board_session, MMIO, root / (name + ".hwh"))
    print("CH2 采集接口预检:")
    print(json.dumps(mark_preflight, indent=2, ensure_ascii=False))

    mpg1_preflight = inspect_multipoint_capability(board_session, buses)
    print("7010 MPG1 能力预检:")
    print(json.dumps(mpg1_preflight, indent=2, ensure_ascii=False))

    from image_owner import ImageOwner
    from image_service import ImageService
    from web_service import start
    import shutil
    web_workspace=root/"web_workspaces"/session_dir.name
    assert not web_workspace.exists()
    shutil.copytree(candidate/"functional/results",web_workspace,
                    ignore=shutil.ignore_patterns("lab_state.sqlite","service.lock"))
    image_web=ImageService(web_workspace,session_root=board_session.root,assets_root=root/"web_image_assets",
        owner_factory=lambda asset,seed:ImageOwner(board_session,buses,asset,seed,board_lock,candidate))
    web_handle=start(web_workspace,image=image_web,port=8765)
    print("上传图片服务已连接新会话；尚未封存图片计划，未执行CONFIG/ITEM、未发送试验START")
    print("保持内核运行，浏览器打开 http://127.0.0.1:18768/upload")
except BaseException:
    if control_uart.context:control_uart.pause_receiver()
    print("保留现场，不重复执行。证据目录:",board_session.root)
    raise


第3步：单独执行，不要Run All。上传和封存后仍需在网页点击真实发送。

In [ ]:
# 等图片页终态、结果生成完成并下载证据后执行。
if image_web.pilot:
    assert image_web.pilot.join(0), "仍在采集或修复，请等待"
    assert image_web.pilot.status()["state"]!="READY", "先在网页停止未执行项"
web_handle.shutdown()
board_session.close()
print("服务和接收器已关闭，请备份完整会话、namespace_allocations.jsonl及trial_registry")
print(board_session.root)
